# B. GEOSPATIAL PYTHON TEST

For this python test, I will be working with data from the Active Major Construction permits in New York City. The idea is to not only analyze the data but also to visualize it through graphs and charts in order to communicate the findings in a more effective way. 

### 0. INSTALL THE NECESSARY PACKAGES FOR THE ANALYSIS

To test it locally we must follow the following steps:

1. Create a virtual environment to install the packages to run the notebook.

**Run:** python -m venv .venv

2. Activate the environment.

**Run:** .venv\Scripts\Activate

3. Install the packages required.

**Run:** pip install -r requirements.txt

4. Launch the Jupyter Notebook.

**Run:** jupyter notebook

In [ ]:
import pandas as pd
import numpy as np
from geopy.geocoders import Nominatim
from pathlib import Path

### 1. LOAD THE DATA

First, we load the dataset from the project's data folder (previously downloaded from the DOB NYC Open Data Website). With approximately 4 million rows and 60 columns, loading the data may take some time.

In [ ]:
Path('data').is_dir()

if (Path.cwd() / 'data' / 'Building_permits_NYC.csv').exists():
    print("File exists")

NameError: name 'Path' is not defined

In [80]:
dataset = pd.read_csv("data/Building_Permits_NYC.csv")

C:\Users\jan10\AppData\Local\Temp\ipykernel_10640\1582457802.py:1: DtypeWarning: Columns (8: Block, 9: Lot, 10: Community Board, 31: Permittee's Phone #, 33: Permittee's License #, 36: HIC License, 47: Owner's House #, 48: Owner's House Street Name, 49: Owner’s House City, 50: Owner’s House State, 51: Owner’s House Zip Code) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv("data/Building_Permits_NYC.csv")


Once we have read the data, we might want to make sure that we can access it. We can use the "head" function to show the first obervations (I decided that 10 would be enough to get familiarized with the data available).

In [81]:
dataset.head(10)

,BOROUGH,Bin #,House #,Street Name,Job #,Job doc. #,Job Type,Self_Cert,Block,Lot,...,Owner’s House State,Owner’s House Zip Code,PERMIT_SI_NO,LATITUDE,LONGITUDE,COUNCIL_DISTRICT,CENSUS_TRACT,BBL,NTA_NAME,DOBRunDate
0,BROOKLYN,3185434,60,BAY 34 ST,340733647,1,A2,Y,6861.0,67.0,...,NY,11214.0,3765466,40.599043,-73.992006,47.0,300.0,NaN,Bensonhurst East,06/18/2020
1,QUEENS,4623172,117,BEACH 215 STREET,421538731,1,NB,N,16350.0,400.0,...,NY,11697.0,3765467,40.554860,-73.923824,32.0,91601.0,NaN,Breezy Point-Belle Harbor-Rockaway Park-Broad ...,06/18/2020
2,STATEN ISLAND,5174573,620,RICHMOND TERRACE,510113501,1,NB,N,62.0,56.0,...,NY,11201.0,3765795,40.645602,-74.092344,49.0,81.0,NaN,West New Brighton-New Brighton-St. George,06/18/2020
3,BRONX,2010402,1323,CHISHOLM ST,220723885,1,DM,N,2971.0,36.0,...,NY,11205.0,3765813,40.830992,-73.895698,16.0,153.0,NaN,Crotona Park East,06/18/2020
4,MANHATTAN,1014387,1,PENN PLAZA,123734717,2,A2,Y,783.0,70.0,...,NY,10019.0,3623536,40.751098,-73.992926,3.0,101.0,NaN,Midtown-Midtown South,07/29/2020
5,STATEN ISLAND,5130986,1000,WEST SERVICE ROAD,510113315,1,NB,N,5900.0,500.0,...,NaN,NaN,3905970,40.571144,-74.195327,51.0,228.0,NaN,New Springville-Bloomfield-Travis,05/12/2022
6,STATEN ISLAND,5130986,1000,WEST SERVICE ROAD,510113315,1,NB,N,5900.0,500.0,...,NaN,NaN,3905979,40.571144,-74.195327,51.0,228.0,NaN,New Springville-Bloomfield-Travis,05/12/2022
7,BRONX,2078937,39,PENNYFIELD CAMP,240240565,1,A2,Y,5519.0,169.0,...,NY,10465.0,3766014,40.821766,-73.806152,13.0,138.0,NaN,Schuylerville-Throgs Neck-Edgewater Park,06/23/2020
8,BROOKLYN,3007326,420,UNION STREET,302576923,1,A2,Y,437.0,1.0,...,NY,11101.0,3766142,40.680371,-73.990676,39.0,75.0,NaN,Carroll Gardens-Columbia Street-Red Hook,06/19/2020
9,QUEENS,4100768,31-25,DOWNING STREET,421661320,3,A2,N,4406.0,28.0,...,NY,11354.0,3766228,40.767755,-73.835808,20.0,869.0,NaN,Flushing,06/19/2020


We can also get a first impression of the different data types we have in our data:

In [82]:
dataset.dtypes.value_counts()

str        41
object      7
float64     7
int64       5
Name: count, dtype: int64

Most columns contain text or string values, indicating that the dataset is primarily qualitative, with fewer quantitative variables.

Before examining data types, missing values, and potential outliers in detail, we can narrow the dataset to match the project's objectives. We retain only **active permits** classified as **New Building (NB)** or **Major Alteration Type 1 (A1)**. This filtering step excludes inactive permits and unrelated job types, allowing us to focus on ongoing major construction activity across New York City.

We will start by filtering the dataset using the following idea:

Give me all the recors where the `job type`is equal to **NB** and **A1**.

In [83]:
counts_job_type = dataset["Job Type"].value_counts()

Once we filter the dataset we should be geting around a million records left (NB + A1)

In [84]:
total_counts =  np.sum([counts_job_type["NB"], counts_job_type["A1"]])

total_counts

np.int64(1006863)

We should be seeing **1,006,863** records in total after filtering the dataset.

In [85]:
dataset_subset = dataset[(dataset["Job Type"] == "NB") | (dataset["Job Type"] == "A1")] 

Now we can also remove of some of the columns that we won't need for our analysis.

1. COLUMNS WITH MISSING VALUES

In [86]:
na_sum = dataset_subset[:].isnull().sum()

na_sum.sort_values(ascending=False)

Oil Gas                             1006863
HIC License                          994121
Site Safety Mgr Business Name        992797
Special District 2                   986231
Site Safety Mgr's First Name         984605
Site Safety Mgr's Last Name          984604
Owner’s House Zip Code               984139
Owner's House Street Name            984079
Owner's House #                      984075
Owner’s House City                   984066
Owner’s House State                  984064
Permittee's Other Title              951266
Special District 1                   914428
Permit Subtype                       799300
Work Type                            576189
Act as Superintendent                522715
Superintendent Business Name         427435
Superintendent First & Last Name     408019
Self_Cert                            366870
Owner's Business Name                308841
Residential                          229845
Permittee's License Type              56659
Permittee's License #           

Several columns contain substantial amounts of missing data, with some having more missing values than valid entries. I will assess their relevance to the analysis and remove columns with excessive missingness that do not provide meaningful information for the project's objectives.

In [87]:
dataset_subset_2 = dataset_subset.drop(columns=["Oil Gas", "HIC License", "Site Safety Mgr Business Name", "Special District 2", "Site Safety Mgr's First Name", 
"Site Safety Mgr's Last Name", "Owner’s House Zip Code", "Owner's House Street Name", "Owner's House #", "Owner’s House City", 
"Owner’s House State", "Permittee's Other Title", "Special District 1","Permit Subtype", "Act as Superintendent", "Superintendent Business Name",
"Superintendent First & Last Name", "Self_Cert", "Owner's Business Name",  "BBL","Block", "Lot", "Community Board","Owner's Business Type", "Permittee's License Type", "Permittee's Phone #", "Residential", "Work Type",
"Permittee's License #", "Site Fill", "Non-Profit", "Permittee's Business Name"])

In [88]:
dataset_subset_2

,BOROUGH,Bin #,House #,Street Name,Job #,Job doc. #,Job Type,Zip Code,Bldg Type,Permit Status,...,Permittee's Last Name,Owner's First Name,Owner's Last Name,PERMIT_SI_NO,LATITUDE,LONGITUDE,COUNCIL_DISTRICT,CENSUS_TRACT,NTA_NAME,DOBRunDate
1,QUEENS,4623172,117,BEACH 215 STREET,421538731,1,NB,11697.0,1.0,ISSUED,...,NOLAN,CHRISTOPHER,NORTON,3765467,40.554860,-73.923824,32.0,91601.0,Breezy Point-Belle Harbor-Rockaway Park-Broad ...,06/18/2020
2,STATEN ISLAND,5174573,620,RICHMOND TERRACE,510113501,1,NB,10301.0,2.0,ISSUED,...,ROME,BEN,ATKINS,3765795,40.645602,-74.092344,49.0,81.0,West New Brighton-New Brighton-St. George,06/18/2020
5,STATEN ISLAND,5130986,1000,WEST SERVICE ROAD,510113315,1,NB,10309.0,2.0,ISSUED,...,DIEKHAUS,GLENN,BRUE (FOR NYC DDC),3905970,40.571144,-74.195327,51.0,228.0,New Springville-Bloomfield-Travis,05/12/2022
6,STATEN ISLAND,5130986,1000,WEST SERVICE ROAD,510113315,1,NB,10309.0,2.0,ISSUED,...,DIEKHAUS,GLENN,BRUE (FOR NYC DDC),3905979,40.571144,-74.195327,51.0,228.0,New Springville-Bloomfield-Travis,05/12/2022
12,QUEENS,4173769,264-52,73 AVE,421600921,1,NB,11004.0,1.0,ISSUED,...,OZKAL,DAVID,MANESH,3906890,40.754576,-73.715416,23.0,155102.0,Glen Oaks-Floral Park-New Hyde Park,05/17/2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3990887,BROOKLYN,3346230,229,MAPLE STREET,300199354,1,NB,11225.0,2.0,ISSUED,...,MALUENDA,DAVIDS,MANESH,999981,40.660701,-73.952816,40.0,800.0,Prospect Lefferts Gardens-Wingate,10/10/2026
3990893,QUEENS,4525783,88-21,170 STREET,401732167,1,NB,11432.0,1.0,ISSUED,...,ATARIEN,JOSEPH,ATARIEN,999987,40.710120,-73.791458,27.0,462.0,Jamaica,10/10/2026
3990901,MANHATTAN,1061075,187,EDGECOMB AVENUE,103960607,1,A1,10030.0,2.0,RE-ISSUED,...,MILLER,PETER,SPINA,999994,40.822391,-73.944456,9.0,227.0,Hamilton Heights-Sugar Hill,10/10/2026
3990905,BROOKLYN,3392276,2103,57 STREET,302065002,1,NB,11204.0,1.0,ISSUED,...,PERSAUD,Yoel,Gruber,999998,40.619611,-73.978647,44.0,468.0,Mapleton-Midwood (West),10/10/2026


The `Work Type` column contains 57% missing values, limiting its usefulness for this analysis. I will retain `Permit Type`, which is almost complete and provides information about the specific work covered by each permit, and exclude `Work Type` from the initial analysis. This decision can be revisited if further analysis requires it.

In [89]:
na_sum_2 = dataset_subset_2[:].isnull().sum()

na_sum_2.sort_values(ascending=False)

Issuance Date             5402
Permit Status             4595
Permittee's Last Name     3831
Permittee's First Name    3823
Bldg Type                 3764
Expiration Date           2056
CENSUS_TRACT              1438
COUNCIL_DISTRICT          1438
LONGITUDE                 1438
LATITUDE                  1438
NTA_NAME                  1438
Owner's First Name         345
Zip Code                   275
Owner's Last Name          243
Job Start Date               6
Street Name                  4
House #                      4
Filing Date                  1
Permit Type                  1
BOROUGH                      0
Filing Status                0
Job #                        0
Job doc. #                   0
Job Type                     0
Bin #                        0
Permit Sequence #            0
PERMIT_SI_NO                 0
DOBRunDate                   0
dtype: int64

After removing columns with excessive missing values or limited relevance to the analysis, I retained 28 columns for further exploration. Next, I assessed missing values in the remaining dataset to determine the most appropriate treatment. Since approximately one million records remain, I decided to remove all of the remaining rows (around 16,000 rows) to end up wth a clean dataset.

In [90]:
dataset_subset_3 = dataset_subset_2.dropna(subset=["Issuance Date", "Permit Status", "Bldg Type", "Owner's First Name", "Owner's Last Name", "Street Name", "Job Start Date", "Permittee's Last Name", "Permittee's First Name", "Zip Code", "Expiration Date", "Permit Type", "Filing Date"])

In [65]:
na_sum_3 = dataset_subset_3[:].isnull().sum()

na_sum_3.sort_values(ascending=False)

NTA_NAME                  1181
LATITUDE                  1181
CENSUS_TRACT              1181
COUNCIL_DISTRICT          1181
LONGITUDE                 1181
Job #                        0
Job Type                     0
Job doc. #                   0
BOROUGH                      0
Bin #                        0
House #                      0
Street Name                  0
Permit Type                  0
Filing Status                0
Permit Status                0
Bldg Type                    0
Zip Code                     0
Permit Sequence #            0
Issuance Date                0
Filing Date                  0
Owner's First Name           0
Permittee's Last Name        0
Permittee's First Name       0
Job Start Date               0
Expiration Date              0
PERMIT_SI_NO                 0
Owner's Last Name            0
DOBRunDate                   0
dtype: int64

I tried using the Nominatim package to geocode some of the addresses anf fill the 1,181 missing values. This API from google maps allows to geocode addresses based on a specific location. Tht's what I have tried but it was taking to long to run so I decided so show how I would do it below and continue with the analysis without running this part.

### -- GEOCODING USING NOMINATIM --

#### 1. CREATING A NEW COLUMN THAT STORES LOCATION IN THE RIGHT FORMAT FOR NOMINATIM

dataset_subset_4 = dataset_subset_3.copy()


dataset_subset_4["Location_geocoder"] = dataset_subset_4["House #"].astype(str) + " " + dataset_subset_4["Street Name"] + ", " + dataset_subset_4["BOROUGH"] + ", New York City, USA"

#### 2. ONLY KEEPING THE ROWS WITH MISSING VALUES INSTEAD OF PASSING 900K ROWS TO THE API

dataset_long_lat_missing = dataset_subset_4[dataset_subset_4["LATITUDE"].isnull() | dataset_subset_4["LONGITUDE"].isnull()]

#### 3. CREATING A FUNCTION THAT ITERATES THROUGH THE ROWS AND USES THE API TO GEOCODE THE ADDRESSES

import time

geolocator = Nominatim(user_agent="wxy_project", timeout=10)



def geocode_location(dataset_long_lat_missing):
    for index, row in dataset_long_lat_missing.iterrows():
        if pd.isnull(row["LATITUDE"]) or pd.isnull(row["LONGITUDE"]):
            location = row["Location_geocoder"]
            geocoded_location = geolocator.geocode(location)
            time.sleep(1)

            if geocoded_location:
                    lat = geocoded_location.latitude
                    lon = geocoded_location.longitude
            else:
                    lat = np.nan
                    lon = np.nan
        else:
            continue

        

        dataset_long_lat_missing.at[index, "LATITUDE"] = lat
        dataset_long_lat_missing.at[index, "LONGITUDE"] = lon

geocode_location(dataset_long_lat_missing)

### CLEAN DATASET FOR ANALYSIS WITH NO EMPTY CELLS

In [92]:
dataset_subset_clean_for_analysis = dataset_subset_3.dropna(subset=["LATITUDE", "LONGITUDE"])

In [94]:
dataset_subset_clean_for_analysis

,BOROUGH,Bin #,House #,Street Name,Job #,Job doc. #,Job Type,Zip Code,Bldg Type,Permit Status,...,Permittee's Last Name,Owner's First Name,Owner's Last Name,PERMIT_SI_NO,LATITUDE,LONGITUDE,COUNCIL_DISTRICT,CENSUS_TRACT,NTA_NAME,DOBRunDate
1,QUEENS,4623172,117,BEACH 215 STREET,421538731,1,NB,11697.0,1.0,ISSUED,...,NOLAN,CHRISTOPHER,NORTON,3765467,40.554860,-73.923824,32.0,91601.0,Breezy Point-Belle Harbor-Rockaway Park-Broad ...,06/18/2020
2,STATEN ISLAND,5174573,620,RICHMOND TERRACE,510113501,1,NB,10301.0,2.0,ISSUED,...,ROME,BEN,ATKINS,3765795,40.645602,-74.092344,49.0,81.0,West New Brighton-New Brighton-St. George,06/18/2020
5,STATEN ISLAND,5130986,1000,WEST SERVICE ROAD,510113315,1,NB,10309.0,2.0,ISSUED,...,DIEKHAUS,GLENN,BRUE (FOR NYC DDC),3905970,40.571144,-74.195327,51.0,228.0,New Springville-Bloomfield-Travis,05/12/2022
6,STATEN ISLAND,5130986,1000,WEST SERVICE ROAD,510113315,1,NB,10309.0,2.0,ISSUED,...,DIEKHAUS,GLENN,BRUE (FOR NYC DDC),3905979,40.571144,-74.195327,51.0,228.0,New Springville-Bloomfield-Travis,05/12/2022
12,QUEENS,4173769,264-52,73 AVE,421600921,1,NB,11004.0,1.0,ISSUED,...,OZKAL,DAVID,MANESH,3906890,40.754576,-73.715416,23.0,155102.0,Glen Oaks-Floral Park-New Hyde Park,05/17/2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3990887,BROOKLYN,3346230,229,MAPLE STREET,300199354,1,NB,11225.0,2.0,ISSUED,...,MALUENDA,DAVIDS,MANESH,999981,40.660701,-73.952816,40.0,800.0,Prospect Lefferts Gardens-Wingate,10/10/2026
3990893,QUEENS,4525783,88-21,170 STREET,401732167,1,NB,11432.0,1.0,ISSUED,...,ATARIEN,JOSEPH,ATARIEN,999987,40.710120,-73.791458,27.0,462.0,Jamaica,10/10/2026
3990901,MANHATTAN,1061075,187,EDGECOMB AVENUE,103960607,1,A1,10030.0,2.0,RE-ISSUED,...,MILLER,PETER,SPINA,999994,40.822391,-73.944456,9.0,227.0,Hamilton Heights-Sugar Hill,10/10/2026
3990905,BROOKLYN,3392276,2103,57 STREET,302065002,1,NB,11204.0,1.0,ISSUED,...,PERSAUD,Yoel,Gruber,999998,40.619611,-73.978647,44.0,468.0,Mapleton-Midwood (West),10/10/2026
